In [11]:
# load full:

# Create Source Data (Day 1)

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Load 1").getOrCreate()

datas = ([1 , "manas" , 100000],
         [2,"vaish" , 200000])
coll = ["id" , "name" , "salary"]

df= spark.createDataFrame(datas , coll)

destination = "/content/load_table"

df.write.mode("overwrite").parquet(destination)
df.show()



+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|manas|100000|
|  2|vaish|200000|
+---+-----+------+



In [10]:
# Modify Source Data (Day 2)

secondary_datas = ([1 , "manas" , 100000],
         [2,"vaish" , 300000],
         [3, "arjun" , 500000])

cols_sec = ["id" , "name" , "salary"]

# Full Load Again (Overwrite Destination)

sec_df = spark.createDataFrame(secondary_datas , cols_sec)

sec_df.write.mode("overwrite").parquet(destination)

print("Destination After Day 2 Full Load")

spark.read.parquet(destination).show()


Destination After Day 2 Full Load
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  2|vaish|300000|
|  3|arjun|500000|
|  1|manas|100000|
+---+-----+------+



In [ ]:
# 31.2 Incremental Load---------------------------

In [24]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Load 1").getOrCreate()

datas = ([1 , "manas" , 100000],
         [2,"vaish" , 200000])

coll = ["id" , "name" , "salary"]

df= spark.createDataFrame(datas , coll)

destination = "/content/Incremental"
df.write.mode("overwrite").saveAsTable("employee_table")


print("After Day 1 Load")
spark.sql("SELECT * FROM employee_table").show()

After Day 1 Load
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  2|vaish|200000|
|  1|manas|100000|
+---+-----+------+



In [25]:
#  Modify Source Data (Day 2)

secondary_datas = ([1 , "manas" , 100000],
         [2,"vaish" , 300000],
         [3, "arjun" , 500000])

sec_df = spark.createDataFrame(secondary_datas , coll)

In [26]:
# load incremental logic /
exisitng_df = spark.table("employee_table")

new_df = sec_df.join(exisitng_df , "id" , "left_anti")

new_df.write.mode("append").saveAsTable("employee_table")

print("After Incremental Load")
spark.sql("SELECT * FROM employee_table").show()

After Incremental Load
+---+-----+------+
| id| name|salary|
+---+-----+------+
|  2|vaish|200000|
|  1|manas|100000|
|  3|arjun|500000|
+---+-----+------+



In [41]:
# 31.3 Snapshot Load -----------------------

spark.sql("DROP TABLE IF EXISTS employee_snapshot")


DataFrame[]

In [45]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("31.3 Snapshot Load").getOrCreate()

data = ([1 , "manas" , 100000],
         [2,"vaish" , 200000])

cols = ["id" , "name" , "salary"]

df = spark.createDataFrame(data , cols)
df.show()

destination = "/content/SnapShot"


new_df = df.withColumn("snapshot_date" , lit("2026-02-28"))

new_df.write.mode("append").parquet("employee_snapshot")

spark.read.parquet("employee_snapshot").show()

#  load now


+---+-----+------+
| id| name|salary|
+---+-----+------+
|  1|manas|100000|
|  2|vaish|200000|
+---+-----+------+

+---+-----+------+-------------+
| id| name|salary|snapshot_date|
+---+-----+------+-------------+
|  2|vaish|200000|   2026-02-28|
|  1|manas|100000|   2026-02-28|
|  2|vaish|200000|   2026-02-28|
|  1|manas|100000|   2026-02-28|
+---+-----+------+-------------+



In [46]:

data = ([1 , "manas" , 100000],
         [2,"vaish" , 300000],
        [3,"Santner" , 500000])

cols = ["id" , "name" , "salary"]

df = spark.createDataFrame(data , cols)

snap_shot_df = df.withColumn("snapshot_date" , lit("2026-03-01"))

snap_shot_df.write.mode("append").parquet("employee_snapshot")

spark.read.parquet("employee_snapshot").show()

+---+-------+------+-------------+
| id|   name|salary|snapshot_date|
+---+-------+------+-------------+
|  2|  vaish|300000|   2026-03-01|
|  3|Santner|500000|   2026-03-01|
|  2|  vaish|200000|   2026-02-28|
|  1|  manas|100000|   2026-02-28|
|  2|  vaish|200000|   2026-02-28|
|  1|  manas|100000|   2026-02-28|
|  1|  manas|100000|   2026-03-01|
+---+-------+------+-------------+

